# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [8]:
# 1. 충돌을 일으킨 패키지 및 기존 PyTorch 삭제
!pip uninstall -y torch torchvision torchaudio

# 2. Unsloth 설치 (이때 기본 torch가 묻어서 설치될 수 있음)
!pip install unsloth

# 3. 처음 환경 설정에 있던 CUDA 12.8 기반 PyTorch로 덮어쓰기 (강제 재설치)
!pip install --pre torch torchvision torchaudio --force-reinstall --index-url https://download.pytorch.org/whl/nightly/cu128

In [9]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [10]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

In [11]:
!pip install --upgrade "torchvision>=0.26.0"

# 라이브러리, 데이터, 설정

In [12]:
!pip install wandb -q

In [13]:
import os, random, math
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# scikit-learn 의존성 보장 (아래 resample import 에러 방지)
try:
    import sklearn  # noqa: F401
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
from sklearn.utils import resample
from dataclasses import dataclass
from typing import Any
!pip install -q --upgrade --force-reinstall "wandb" "protobuf>=4.25.3,<6"
import importlib, sys, subprocess

try:
    import wandb
except Exception:
    # wandb/protobuf 충돌 복구
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
        "wandb", "protobuf>=4.25.3,<6"
    ])
    importlib.invalidate_caches()
    for m in list(sys.modules):
        if m.startswith("wandb") or m.startswith("google.protobuf"):
            sys.modules.pop(m, None)
import wandb
from unsloth import FastVisionModel 
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

# ==========================================
# 💡 [W&B 팀 공용 API 키 설정]
# ==========================================
os.environ["WANDB_API_KEY"] = "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"
wandb.login()

# ==========================================
# 💡 [팀원 4 전용 설정] 패딩 O / 오버샘플링 X
# ==========================================
EXP_NAME = "Exp4_Padding_No_Oversample"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct" # 공식 지원 비전 모델명
USE_OVERSAMPLING = False               # 💡 뻥튀기 없이 원본 분포 그대로 학습
PREPROCESSING = "pad"                  # 💡 여백을 검은색으로 채워 비율 유지
MAX_CAP = 400                          # (오버샘플링 False라 무시되지만 공통 폼 유지)

# 공통 환경 설정
Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 672
MAX_NEW_TOKENS = 8
SEED = 42

random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"✅ Device: {device} | Experiment: {EXP_NAME}")

✅ Device: cuda | Experiment: Exp4_Padding_No_Oversample


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [14]:
# 1. 층화 추출을 통한 Valid 셋 분리 (answer + 질문유형 복합 key)
from sklearn.model_selection import train_test_split

train_full_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 정답이 비어있는 불량 데이터 제거
train_full_df = train_full_df.dropna(subset=['answer']).reset_index(drop=True)

# 질문 유형 분류 함수
def get_qtype(q):
    if '몇 개' in q or '개수' in q or '몇개' in q: return 'count'
    if '재질' in q: return 'material'
    if '색상' in q or '색깔' in q: return 'color'
    if '종류' in q: return 'type'
    return 'other'

# answer + 질문유형 복합 층화 key 생성
train_full_df['strat_key'] = train_full_df['answer'] + '_' + train_full_df['question'].apply(get_qtype)

train_df, valid_df = train_test_split(
    train_full_df, test_size=0.1, stratify=train_full_df['strat_key'], random_state=SEED
)

# 임시 컬럼 제거
train_df = train_df.drop(columns=['strat_key']).reset_index(drop=True)
valid_df = valid_df.drop(columns=['strat_key']).reset_index(drop=True)

print(f'Train: {len(train_df)}개 / Valid: {len(valid_df)}개')
print('Train 정답 분포:', train_df['answer'].value_counts().sort_index().to_dict())
print('Valid 정답 분포:', valid_df['answer'].value_counts().sort_index().to_dict())

def pad_to_square(img):
    max_side = max(img.size)
    return ImageOps.pad(img, (max_side, max_side), color=(0, 0, 0))

def center_crop_square(img):
    min_side = min(img.size)
    return ImageOps.fit(img, (min_side, min_side), centering=(0.5, 0.5))


Train: 4565개 / Valid: 508개
Train 정답 분포: {'a': 1137, 'b': 1138, 'c': 1179, 'd': 1111}
Valid 정답 분포: {'a': 126, 'b': 127, 'c': 132, 'd': 123}


In [15]:
# 모델 로드 및 LoRA 설정
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    load_in_8bit=False,
    device_map='cuda:0',
)
model = FastVisionModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head']
)
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
print('모델 로드 완료')


==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260401+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:42<00:00, 21.15s/it]


모델 로드 완료


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [16]:
SYSTEM_INSTRUCT = (
    '당신은 재활용 분류 전문가입니다. '
    '이미지를 분석할 때 선택지의 글자 수나 질문의 키워드에 의존하지 마세요. '
    '반드시 이미지에서 보이는 물체의 형태, 재질, 색상 등 시각적 증거만을 근거로 판단하세요. '
    '모든 선택지를 시각적 증거와 대조한 후, a, b, c, d 중 정확히 한 글자만 출력하세요. '
    '설명, 부호, 추가 텍스트는 절대 출력하지 마세요.'
)

def build_mc_prompt(question, a, b, c, d, gold_answer=None):
    prompt = (
        f'{question}\n'
        f'(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'
        '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.'
    )
    return prompt, gold_answer

# 하위 호환성 유지
build_shuffled_mc_prompt = build_mc_prompt


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [17]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        q = str(row['question'])
        a, b, c, d = str(row['a']), str(row['b']), str(row['c']), str(row['d'])
        gold_answer = str(row['answer']).strip().lower() if self.train else None
        user_text, new_gold = build_mc_prompt(q, a, b, c, d, gold_answer)

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
        ]
        if self.train:
            messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': new_gold}]})

        return {'messages': messages, 'image': img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def _find_response_start(self, input_ids_list):
        """마지막 <|im_start|> 이후 개행까지 스킵 → assistant 답변 시작 위치 반환"""
        tokenizer = self.processor.tokenizer
        im_start = tokenizer.convert_tokens_to_ids('<|im_start|>')
        nl_ids = tokenizer.encode('\n', add_special_tokens=False)
        nl_id = nl_ids[0] if nl_ids else None

        # 마지막 <|im_start|> 위치 탐색
        for pos in range(len(input_ids_list) - 1, -1, -1):
            if input_ids_list[pos] == im_start:
                # <|im_start|> 이후 첫 개행 위치 찾기
                for j in range(pos + 1, min(pos + 6, len(input_ids_list))):
                    if nl_id and input_ids_list[j] == nl_id:
                        return j + 1  # 개행 다음이 실제 답변 토큰
                return pos + 3  # fallback
        return 0

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            texts.append(self.processor.apply_chat_template(
                sample['messages'], tokenize=False, add_generation_prompt=False))
            images.append(sample['image'])

        enc = self.processor(text=texts, images=images, padding=True, return_tensors='pt')

        if self.train:
            labels = enc['input_ids'].clone()

            # 패딩 마스킹
            pad_id = self.processor.tokenizer.pad_token_id
            if pad_id is not None:
                labels[labels == pad_id] = -100

            # 각 샘플: assistant 응답 시작 전까지 -100 (이미지/질문/시스템 프롬프트 제외)
            for b_idx in range(labels.shape[0]):
                resp_start = self._find_response_start(enc['input_ids'][b_idx].tolist())
                labels[b_idx, :resp_start] = -100

            enc['labels'] = labels
        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [18]:
# Cell 13에서 층화 추출된 train_df, valid_df를 DataLoader로 변환
train_ds = VQAMCDataset(train_df, processor, train=True)
valid_ds = VQAMCDataset(valid_df, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                          collate_fn=DataCollator(processor, True), num_workers=0)

print(f'train_loader: {len(train_loader)}스텝 / valid_loader: {len(valid_loader)}스텝')


train_loader: 4565스텝 / valid_loader: 508스텝


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
import math, os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup

model = model.to(device)
GRAD_ACCUM = 4
EPOCHS = 1

no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay) and p.requires_grad], 'weight_decay': 0.03},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay) and p.requires_grad], 'weight_decay': 0.0},
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=6e-5)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps)
scaler = torch.amp.GradScaler('cuda', enabled=True)
global_step = 0

print(f'학습 시작 (총 {EPOCHS} Epoch, {num_training_steps} steps)')
for epoch in range(EPOCHS):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Train]', unit='batch')

    model.train()
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            logits = outputs.logits[..., :-1, :].contiguous()
            shift_labels = batch['labels'][..., 1:].contiguous()
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
                label_smoothing=0.05
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            progress_bar.set_postfix({'loss': f'{running:.3f}', 'lr': f'{scheduler.get_last_lr()[0]:.1e}'})
            running = 0.0

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f'Epoch {epoch+1} [Valid]', unit='batch'):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f'[Epoch {epoch+1}] Valid Loss: {val_loss/val_steps:.4f}')
    model.train()

SAVE_DIR = os.path.join(os.getcwd(), 'qwen3_vl_finetuned')
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f'모델 저장 완료: {SAVE_DIR}')


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_10656\3290388149.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)
Epoch 1 [Train]:   0%|          | 0/4565 [00:00<?, ?batch/s]C:\Users\SSAFY\AppData\Local\Temp\ipykernel_10656\3290388149.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Epoch 1 [Train]:   0%|          | 1/4565 [00:31<39:46:58, 31.38s/batch]

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from collections import Counter
from unsloth import FastVisionModel

# 학습된 모델을 추론 모드로 전환 (재로드 없이 기존 모델 사용)
FastVisionModel.for_inference(model)
model.eval()

TTA_TRIALS = 3  # 선지 순서를 3회 셔플 후 다수결

def extract_choice(text):
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines: return 'a'
    last = lines[-1]
    if last in ['a', 'b', 'c', 'd']: return last
    for tok in last.split():
        if tok in ['a', 'b', 'c', 'd']: return tok
    return 'a'

def run_inference(img, question, choices_ordered):
    user_text = (
        f'{question}\n'
        f'(a) {choices_ordered[0]}\n(b) {choices_ordered[1]}\n'
        f'(c) {choices_ordered[2]}\n(d) {choices_ordered[3]}\n\n'
        '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.'
    )
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
        {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors='pt').to(device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    return processor.batch_decode(out_ids, skip_special_tokens=True)[0]

label_to_idx = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
preds = []

for i in tqdm(range(len(test_df)), desc=f'TTA Inference (x{TTA_TRIALS})', unit='sample'):
    row = test_df.iloc[i]
    img = Image.open(row['path']).convert('RGB')
    if PREPROCESSING == 'pad':
        img = pad_to_square(img)
    elif PREPROCESSING == 'crop':
        img = center_crop_square(img)

    orig_choices = [str(row['a']), str(row['b']), str(row['c']), str(row['d'])]
    votes = []

    for t in range(TTA_TRIALS):
        order = [0, 1, 2, 3]
        if t > 0:
            random.shuffle(order)
        shuffled_choices = [orig_choices[j] for j in order]
        pred_letter = extract_choice(run_inference(img, str(row['question']), shuffled_choices))
        pred_pos = label_to_idx.get(pred_letter, 0)
        original_idx = order[pred_pos]
        votes.append(['a', 'b', 'c', 'd'][original_idx])

    preds.append(Counter(votes).most_common(1)[0][0])

# 제출 파일 저장
import glob
base = 'submission_TTA'
existing = glob.glob(f'{base}*.csv')
filename = f'{base}_v{len(existing)+1}.csv'
submission = pd.DataFrame({'id': test_df['id'], 'answer': preds})
submission.to_csv(filename, index=False)
print(f'저장 완료: {filename}')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\SSAFY\AI_Challenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0403 10:08:33.402000 40768 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0403 10:08:33.448000 40768 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260401+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.93s/it]


NameError: name 'tqdm' is not defined